# Complex rules opposite to their simpler rules
Complex rules that passed p, then FDR, then point the other way (attract vs avoid) from all their simpler rules in the same field.

In [ ]:
import os
import sys

import pandas as pd

sys.path[:0] = [os.path.abspath("scripts"), os.path.abspath(".."), os.path.abspath("../differential_rules/scripts")]
import complex_investigation as ci
import complex_investigation_vis as vis
import data_helper as dh
import significance_histograms_vis as sh

In [ ]:
MAX_FDR = 0.05
P_CUT = 0.05
PREFIX = "opposite"
LABELS = ["opposite to all simpler rules"] + sh.group_labels(P_CUT, MAX_FDR)
COLORS = [sh.OPPOSITE] + sh.COLORS
P_LINE = (P_CUT, f"p = {P_CUT}")
N_EXAMPLES = 5
KINDS = {"attracts": "attracts (lift > 1)", "avoids": "avoids (lift < 1)"}
DIRECTIONS = {"complex-antecedents": "many to one", "complex-consequents": "one to many"}

In [ ]:
rules = dh._read_rules(dh.RESULT_CSV_PATH, rule_max_items=4)
complex_rules = rules[dh.n_items(rules) > dh.PAIRWISE_ITEMS]
raw_rules = pd.read_csv(os.path.join(dh.RESULT_CSV_PATH, "results_CN_RAW.csv"),
                        usecols=["FOV", "Antecedents", "Consequents", "len_ant", "len_con",
                                 "Support", "Confidence", "Lift", "Leverage", "Conviction"])
comparisons = pd.read_csv(os.path.join(dh.RESULT_CSV_PATH, dh.COMPARISONS_FILENAME),
                          usecols=["FOV", "rule_idx", "simpler_rule", "Kind"])
cells, metadata, _ = dh.load_spatial_data()

In [ ]:
shape = sh.shape_and_size(complex_rules)
opposite = ci.opposite_to_simpler(rules, comparisons, raw_rules)[complex_rules.index]
groups = sh.with_opposite(sh.significance(complex_rules, P_CUT, MAX_FDR), opposite)
opposite_rules = complex_rules[opposite]
size = dh.n_items(opposite_rules).astype(str) + " items"
passed = opposite_rules[groups[opposite].eq(0)]
picks, prepared = ci.opposite_examples(passed, raw_rules, comparisons, metadata, N_EXAMPLES)

## 1. How many pass each step

In [ ]:
stages = pd.DataFrame({"all": True,
                       f"p ≤ {P_CUT}": groups.between(0, 2),
                       f"FDR ≤ {MAX_FDR}": groups.between(0, 1),
                       "opposite": groups.eq(0)})
per_fov = stages.groupby([complex_rules.FOV, shape], observed=True).sum()

display(stages.groupby(shape, observed=True).sum())
display(per_fov.groupby(level=1, observed=True).describe().drop(columns=["count", "std"], level=1).T.round(1))

## 2. Rule p-values

In [ ]:
sh.histograms(sh.counts(dh.rule_p_value(complex_rules), shape, groups, sh.P_BINS),
              sh.P_BINS, LABELS, "p-value", "rules", f"{PREFIX}_p_values", P_LINE, COLORS)

## 3. Opposite rules
One pie per number of items. The own test covers all opposite rules; the other two show all of them, then only those that passed FDR.

### Their own test

In [ ]:
tests = pd.crosstab(size, sh.significance(opposite_rules, P_CUT, MAX_FDR)).reindex(columns=range(3), fill_value=0)
sh.pies(tests.set_axis(sh.group_labels(P_CUT, MAX_FDR), axis=1), sh.COLORS, f"{PREFIX}_tests")

### Attract or avoid

In [ ]:
for name, here in {"all": opposite_rules, "passed FDR": passed}.items():
    sh.pies(pd.crosstab(size[here.index], here.Kind.map(KINDS)), [sh.vh.ATTRACTION, sh.vh.AVOIDANCE],
            f"{PREFIX}_kinds_{name.replace(' ', '_')}", f"Opposite rules, {name}")

### Many to one or one to many

In [ ]:
for name, here in {"all": opposite_rules, "passed FDR": passed}.items():
    sh.pies(pd.crosstab(size[here.index], dh.rule_shape(here).map(DIRECTIONS)), sh.vh.SHARE_COLORS,
            f"{PREFIX}_directions_{name.replace(' ', '_')}", f"Opposite rules, {name}")

## 4. Examples
The N_EXAMPLES lowest-FDR opposite rules that passed FDR, for each number of items and shape, next to one of their simpler rules in the same field.

In [ ]:
for index in picks.index:
    vis.show_rule(picks, index, picks, prepared, cells, metadata, ci.Config(), PREFIX)